# 纯 JAX 实现 VMC 梯度下降（朴素梯度下降法）

## 方法说明

本 notebook 使用修复后的 force-based 梯度计算，采用**朴素梯度下降法**（不使用 SR 自然梯度）。

## 梯度公式

Force-based 梯度：
$$
\nabla \langle E \rangle = \langle (E_{loc} - \langle E \rangle) \nabla \log \psi \rangle
$$

参数更新规则：
$$
\theta_{t+1} = \theta_t - \eta \nabla \langle E \rangle
$$

其中 $\eta = 0.01$ 是学习率。

## 实验设置

- 迭代次数：400
- 学习率：0.01
- 优化器：SGD（朴素梯度下降）
- **不使用 SR 预处理**

In [ ]:
# ===================== 环境配置 =====================
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import nnx
import optax
from functools import partial
from jax import flatten_util

print(f"JAX version: {jax.__version__}")
print(f"NetKet version: {nk.__version__}")

In [ ]:
# ===================== 1. H₂ 分子定义 & FCI 基准 =====================
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

# FCI 精确基准
cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()
print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能：{exc:.4f} eV")

In [ ]:
# ===================== 2. NetKet 哈密顿量和采样器 =====================
ha = nkx.operator.from_pyscf_molecule(mol)

hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1,1),
)

edges = [(0, 1), (2, 3)]
g = nk.graph.Graph(edges=edges)
single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
sampler = nk.sampler.MetropolisSampler(hi, rule=single_rule, n_chains=16, sweep_size=32)

print(f"Hilbert space size: {hi.size}")
# print(f"Hamiltonian terms: {ha.n_operators}")

In [ ]:
# ===================== 3. 神经网络 Ansatz =====================
class SingleStateAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals: int, hidden_dim=16, *, rngs: nnx.Rngs):
        super().__init__()
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x):
        h = nnx.tanh(self.linear1(x.astype(complex)))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        return jnp.squeeze(out)

In [ ]:
# ===================== 4. 包装模型为 machine 函数 =====================
def create_machine(model: nnx.Module):
    """将 Flax NNX 模型包装为 NetKet 风格的 machine 函数"""
    graphdef, state = nnx.split(model)
    
    @jax.jit
    def machine(params, sigma):
        m = nnx.merge(graphdef, params)
        return m(sigma)
    
    return machine, graphdef, state

In [ ]:
# ===================== 5. 纯 JAX 实现的 force-based 梯度计算 =====================
@partial(jax.jit, static_argnames=("machine",))
def compute_local_energies(machine, params, sigma):
    """
    计算局部能量 E_loc(σ) = Σ_η H(σ→η) ψ(η)/ψ(σ)
    
    这对应 NetKet 的 local_value_kernel
    """
    eta, H_eta = ha.get_conn_padded(sigma)
    logpsi_sigma = machine(params, sigma)
    logpsi_eta = machine(params, eta)
    logpsi_sigma = jnp.expand_dims(logpsi_sigma, -1)
    return jnp.sum(H_eta * jnp.exp(logpsi_eta - logpsi_sigma), axis=-1)


def statistics(x):
    """计算样本统计量"""
    mean = jnp.mean(x)
    var = jnp.var(x)
    return mean, jnp.sqrt(var / x.shape[0])


@partial(jax.jit, static_argnames=("machine",))
def forces_expect_hermitian(machine, params, sigma):
    """
    核心：复刻 NetKet 的 forces_expect_hermitian 函数
    
    使用 force-based 梯度计算：
    ∇⟨E⟩ = ⟨(E_loc - ⟨E⟩) ∇log ψ⟩
    
    关键：对于复数值网络，使用 holomorphic=True
    """
    # 1. 计算局部能量
    O_loc = compute_local_energies(machine, params, sigma)
    
    # 2. 统计能量均值
    O_mean, O_std = statistics(O_loc)
    
    # 3. 中心化局部能量
    O_centered = O_loc - O_mean
    
    # 4. 计算 ∇log ψ 对每个样本
    # 使用 jax.grad 计算复数梯度（holomorphic=True）
    def log_psi_single(p, s):
        return machine(p, s)
    
    def compute_grad_for_sample(s):
        return jax.grad(lambda p: log_psi_single(p, s), holomorphic=True)(params)
    
    grad_matrix = jax.vmap(compute_grad_for_sample)(sigma)
    
    # 5. 计算 force-based 梯度
    # grad = ⟨(E_loc - E_mean) ∇log ψ⟩ = (1/N) Σ (E_loc[i] - E_mean) ∇log ψ(σ[i])
    # grad_matrix 已经是 PyTree 结构，每个元素的形状是 (n_samples, ...)
    # 关键修复：O_centered 形状为 (n_samples,)，需要正确广播到梯度张量的每个维度
    # 使用 reshape 将 O_centered 变为 (n_samples, 1, 1, ..., 1) 以匹配梯度张量
    def weight_and_mean(grad_component):
        # grad_component 形状：(n_samples, d1, d2, ...)
        # O_centered 形状：(n_samples,)
        # 需要广播相乘后沿 axis=0 求平均
        weights = O_centered.reshape((O_centered.shape[0],) + (1,) * (grad_component.ndim - 1))
        return jnp.mean(weights * jnp.conj(grad_component), axis=0)
    
    grad = jax.tree_util.tree_map(weight_and_mean, grad_matrix)
    
    return O_mean, O_std, grad

In [ ]:
# ===================== 6. 初始化 =====================
rngs = nnx.Rngs(21)
model = SingleStateAnsatz(4, hidden_dim=12, rngs=rngs)
machine, graphdef, params = create_machine(model)

sampler_state = sampler.init_state(machine, params, seed=1)
optimizer = optax.sgd(learning_rate=0.01)  # 学习率 0.01
opt_state = optimizer.init(params)

# 训练参数
N_ITER = 800  # 迭代 400 次
N_SAMPLES = 1008  # 样本数
USE_SR = False  # 不使用 SR 自然梯度

print("初始化完成!")
n_params, _ = flatten_util.ravel_pytree(params)
print(f"模型参数数量：{n_params.shape[0]}")
print(f"学习率：0.01")
print(f"迭代次数：{N_ITER}")
print(f"使用 SR: {USE_SR}")

In [ ]:
# ===================== 7. 训练循环 =====================
print("\n" + "="*60)
print("开始纯 JAX VMC 训练 (朴素梯度下降法)")
print("="*60)

# 用于记录训练历史
history = {
    'step': [],
    'energy': [],
    'energy_std': [],
    'error': []
}

sampler_state = sampler.init_state(machine,params,seed=21)


for step in range(N_ITER):
    # 1. 采样
    sampler_state = sampler.reset(machine,params,sampler_state)
    
    samples, sampler_state = sampler.sample(
        machine, params, state=sampler_state, 
        chain_length=N_SAMPLES // sampler.n_chains
    )
    samples = samples.reshape(-1, hi.size)
    
    # 2. 计算 force-based 能量和梯度
    energy, energy_std, grad = forces_expect_hermitian(machine, params, samples)
    grad = jax.tree_map(lambda x: x*2, grad)
    
    
    # 3. 应用 SR 自然梯度（本例不使用）
    if USE_SR:
        grad = apply_sr(machine, params, samples, grad, diag_shift=0.1)
    
    # 4. 更新参数（朴素梯度下降）
    updates, opt_state = optimizer.update(grad, opt_state, params)
    params = optax.apply_updates(params, updates)
    
    # 5. 记录历史
    if step % 100 == 0 or step == N_ITER - 1:
        error = jnp.abs(energy.real - E_fcis[0])
        history['step'].append(step)
        history['energy'].append(float(energy.real))
        history['energy_std'].append(float(energy_std))
        history['error'].append(float(error))
        print(f"Step {step:3d} | E: {energy.real:.8f} ± {energy_std:.6f} | FCI: {E_fcis[0]:.8f} | Error: {error:.6f}")

# 最终结果
final_energy, final_std, _ = forces_expect_hermitian(machine, params, samples)
final_error = jnp.abs(final_energy.real - E_fcis[0])
print("\n" + "="*60)
print(f"训练完成!")
print(f"最终能量：{final_energy.real:.8f} ± {final_std:.6f} Ha")
print(f"FCI 基准：{E_fcis[0]:.8f} Ha")
print(f"绝对误差：{final_error:.6f} Ha")
print(f"相对误差：{final_error / jnp.abs(E_fcis[0]) * 100:.4f}%")
print("="*60)

In [ ]:
# ===================== 8. 结果可视化 =====================
import matplotlib.pyplot as plt

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 图 1：能量收敛曲线
ax1.plot(history['step'], history['energy'], 'b-', linewidth=2, label='VMC Energy')
ax1.axhline(y=E_fcis[0], color='r', linestyle='--', linewidth=2, label=f'FCI: {E_fcis[0]:.8f} Ha')
ax1.fill_between(history['step'], 
                 [e - s for e, s in zip(history['energy'], history['energy_std'])],
                 [e + s for e, s in zip(history['energy'], history['energy_std'])], 
                 alpha=0.3, color='blue', label='±std')
ax1.set_xlabel('Iteration Step', fontsize=12)
ax1.set_ylabel('Energy (Ha)', fontsize=12)
ax1.set_title('VMC Energy Convergence (Gradient Descent)', fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

# 图 2：误差收敛曲线
ax2.semilogy(history['step'], history['error'], 'g-', linewidth=2)
ax2.set_xlabel('Iteration Step', fontsize=12)
ax2.set_ylabel('Absolute Error (Ha)', fontsize=12)
ax2.set_title('Absolute Error vs FCI Energy', fontsize=14)
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('vmc_gradient_descent_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n图表已保存为 'vmc_gradient_descent_convergence.png'")

## 总结

### 实验设置
- **优化方法**：朴素梯度下降（SGD）
- **学习率**：0.01
- **迭代次数**：400
- **样本数**：1008
- **SR 预处理**：未使用

### 关键实现

1. **Force-based 梯度计算**：
   - 使用 `jax.grad(..., holomorphic=True)` 计算复数梯度
   - 公式：$\nabla \langle E \rangle = \langle (E_{loc} - \langle E \rangle) \nabla \log \psi \rangle$

2. **参数更新**：
   - 使用 Optax 的 SGD 优化器
   - $\theta_{t+1} = \theta_t - \eta \nabla \langle E \rangle$

3. **与 NetKet 的对比**：
   - 本实现使用 `jax.grad` + `holomorphic=True`
   - NetKet 使用 `nkjax.vjp` + `conjugate=True`
   - 两者在数学上等价，但数值实现可能略有差异

### 预期结果

使用朴素梯度下降法（不使用 SR），收敛速度会比 SR 慢，但实现更简单。
通过 400 次迭代，应该能够接近 FCI 基态能量。

### 参考

- [VMC_jax_native.ipynb](../0504/VMC_jax_native.ipynb) - 原始实现（包含 SR）
- [VMC 梯度计算对比分析.ipynb](./VMC 梯度计算对比分析.ipynb) - 梯度计算详细分析